# submission

add task 1/2/4 notebook outputs as input, run all. internet on

task 3 = scored off task 1 rows

download submission.csv right away, new session wipes it

In [ ]:
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
REL = Path("/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public")
REPO = Path("/tmp/trafficflowbench-public")
OUT = Path("/kaggle/working")


def run(*args):
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    r.check_returncode()
    return r.stdout


if not REPO.exists():
    run("git", "clone", "https://github.com/jacky850/trafficflowbench-public.git", REPO)
print("organizers' repo commit:", run("git", "-C", REPO, "rev-parse", "HEAD").strip())


def stack(parts, dest):
    with open(dest, "w") as out:
        for i, p in enumerate(parts):
            with open(p) as f:
                header = f.readline()
                if i == 0:
                    out.write(header)
                out.writelines(f)


def count_rows(path):
    with open(path) as f:
        return sum(1 for _ in f) - 1

In [ ]:
def find(name):
    hits = sorted(Path("/kaggle/input").rglob(name)) or sorted(OUT.glob(name))
    assert len(hits) == 1, f"{name}: expected one file, found {hits}"
    print(name, "->", hits[0])
    return hits[0]


state, queue, odme = find("state.csv"), find("queue.csv"), find("odme.csv")

EXPECTED = {state: 6_735_795, queue: 174_000, odme: 70_708}
for path, n in EXPECTED.items():
    assert count_rows(path) == n, (path, count_rows(path), n)

In [ ]:
log = run("python", REPO / "src/merge_submissions.py",
          "--state", state, "--queue", queue, "--odme", odme,
          "--key", REL / "submission_key.csv", "--output", OUT / "submission.csv")
assert "Every scored cell was filled" in log, "merge left cells at 0, see WARNING above"

In [ ]:
sub = pd.read_csv(OUT / "submission.csv")
assert len(sub) == 6_980_503, len(sub)
assert list(sub.columns) == ["submission_id", "task", "speed_kmh", "flow_vph", "queue_pred", "path_flow"]
assert not sub.isna().any().any(), "empty cells"
print(sub.groupby("task").size())
print(sub.groupby("task")[["speed_kmh", "flow_vph", "queue_pred", "path_flow"]].mean().round(3))